In [1]:
import torch
import torch.nn as nn
from torch import Tensor

device = "cuda"


In [2]:
from transformers import pipeline

model_id = "openai/clip-vit-base-patch32"
clip_pipeline = pipeline(task="zero-shot-image-classification", 
                         model=model_id, device_map="auto", dtype="auto")
candidate_labels = ["cricket", "ladybug", "spider"]
image_url = "https://homl.info/ladybug"  # a photo of a ladybug on a dandelione

results = clip_pipeline(image_url, candidate_labels=candidate_labels, 
                        hypothesis_template="This is a photo of a {}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [3]:
results

[{'score': 0.9973388314247131, 'label': 'ladybug'},
 {'score': 0.0014530560001730919, 'label': 'spider'},
 {'score': 0.0012080862652510405, 'label': 'cricket'}]

In [4]:
import PIL
import urllib.request
import PIL.Image
from transformers import CLIPProcessor, CLIPModel

clib_processor = CLIPProcessor.from_pretrained(model_id)
clib_model = CLIPModel.from_pretrained(model_id)
image = PIL.Image.open(urllib.request.urlopen(image_url)).convert("RGB")
captions = [f"This is a photo of a {label}." for label in candidate_labels]
inputs = clib_processor(text=captions, images=[image], return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = clib_model(**inputs)

text_features:Tensor = outputs.text_embeds
image_features:Tensor = outputs.image_embeds

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [5]:
text_features

tensor([[-0.0088,  0.0071, -0.0149,  ..., -0.0048,  0.0232, -0.0140],
        [-0.0105,  0.0052,  0.0055,  ..., -0.0053, -0.0052, -0.0556],
        [ 0.0287,  0.0153,  0.0251,  ..., -0.0413,  0.0210, -0.0200]])

In [ ]:
#already l2 nomalized
similarities = image_features @ text_features.T
similarities

tensor([[0.2336, 0.3021, 0.2380]])

In [ ]:
temperature = clib_model.logit_scale.detach().exp()
rescaled_similarities = similarities * temperature
probabilities = torch.nn.functional.softmax(rescaled_similarities, dim=1)
probabilities

tensor([[0.0011, 0.9973, 0.0017]])